# Phase 7: Data Preprocessing

The purpose of this phase is to prepare the feature-engineered dataset for machine-learning models.

The workflow will be:

Load feature-engineered dataset
        ↓
Validate the dataset
        ↓
Convert and sort the date
        ↓
Separate features and target
        ↓
Create chronological train-test split
        ↓
Identify feature types
        ↓
Build preprocessing transformer
        ↓
Fit only on training data
        ↓
Transform training and test data
        ↓
Validate the prepared data
        ↓
Save preprocessing outputs

This phase prepares the feature-engineered Walmart sales dataset for machine learning.

The main tasks include:

- loading and validating the featured dataset;
- separating features from the target variable;
- splitting the observations chronologically;
- identifying categorical, binary, and numerical features;
- encoding categorical variables;
- scaling numerical variables;
- creating a reusable preprocessing pipeline.

The preprocessing steps will be fitted only on the training data to prevent data leakage.

## Import Libraries

The required libraries are imported for data loading, preprocessing, and chronological splitting.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## What these imports do
- Path helps create reliable file paths.
- numpy supports numerical operations.
- pandas loads and manages the dataset.
- ColumnTransformer applies different preprocessing methods to different columns.
- OneHotEncoder converts categorical values into numerical columns.
- StandardScaler puts numerical columns on comparable scales.

## Define Project Paths

The project paths are created so that the notebook can locate the featured dataset and save the processed outputs.

In [2]:
from pathlib import Path

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# Handle the accidental notebooks/notebooks structure
if project_root.name == "notebooks":
    project_root = project_root.parent

prepared_data_path = project_root / "data" / "prepared"
models_path = project_root / "models"

prepared_data_path.mkdir(parents=True, exist_ok=True)
models_path.mkdir(parents=True, exist_ok=True)

print("Current folder:", current_path)
print("Project root:", project_root)
print("Prepared folder:", prepared_data_path)

Current folder: c:\Users\Udoch\walmart-sales-regression\notebooks
Project root: c:\Users\Udoch\walmart-sales-regression
Prepared folder: c:\Users\Udoch\walmart-sales-regression\data\prepared


### This creates a new folder

data/
├── processed/
│   ├── walmart_cleaned.csv
│   └── walmart_featured.csv
└── prepared/

## Validating the Input Dataset

Before loading the dataset, the notebook confirms that the feature-engineered CSV file exists.

In [3]:
# Validating the featured dataset

if not featured_data_path.exists():
    raise FileNotFoundError(
        f"Featured dataset not found at: {featured_data_path}"
    )

print("Featured dataset found successfully.")

NameError: name 'featured_data_path' is not defined

In [ ]:
# Loading the dataset

df = pd.read_csv(featured_data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (6435, 13)


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,week_of_year,quarter,time_index
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,0
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324,2010,2,5,1,0
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368,2010,2,5,1,0
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623,2010,2,5,1,0
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566,2010,2,5,1,0


The number of columns is greater than the original eight columns because feature engineering added date-based features. It is now 13 columns.

In [ ]:
# Inspecting all column names

print("Dataset columns:")

for column in df.columns:
    print("-", column)

Dataset columns:
- store
- date
- weekly_sales
- holiday_flag
- temperature
- fuel_price
- cpi
- unemployment
- year
- month
- week_of_year
- quarter
- time_index


In [ ]:
# Validate the data

# Inspect the data types

df.dtypes

store             int64
date                str
weekly_sales    float64
holiday_flag      int64
temperature     float64
fuel_price      float64
cpi             float64
unemployment    float64
year              int64
month             int64
week_of_year      int64
quarter           int64
time_index        int64
dtype: object

## Converting the Date Column

CSV files usually reload date values as text.

The `date` column is converted back to a pandas datetime type so that the dataset can be sorted and split chronologically.

In [ ]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="raise",
)

print("Date data type:", df["date"].dtype)

Date data type: datetime64[us]


In [ ]:
# Checking the missing values

missing_values = df.isnull().sum()

missing_values[missing_values > 0]

Series([], dtype: int64)

In [ ]:
# displaying the total

print(
    "Total missing values:",
    df.isnull().sum().sum(),
)

Total missing values: 0


In [ ]:
# Checking duplicate store-date records

duplicate_count = df.duplicated(
    subset=["store", "date"]
).sum()

print(
    "Duplicate store-date records:",
    duplicate_count,
)

Duplicate store-date records: 0


## Chronological Sorting

The data must be ordered from the earliest date to the latest date before splitting.

This ensures that the model is trained on earlier records and evaluated on later records, which better represents a real sales-prediction scenario.

In [ ]:
# Sorting the dataset chronologically

df = (
    df
    .sort_values(
        by=["date", "store"]
    )
    .reset_index(drop=True)
)

df.head()

,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,week_of_year,quarter,time_index
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,0
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324,2010,2,5,1,0
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368,2010,2,5,1,0
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623,2010,2,5,1,0
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566,2010,2,5,1,0


In [ ]:
# Confirming the order

print(
    "Earliest date:",
    df["date"].min(),
)

print(
    "Latest date:",
    df["date"].max(),
)

print(
    "Dates are sorted:",
    df["date"].is_monotonic_increasing,
)

Earliest date: 2010-02-05 00:00:00
Latest date: 2012-10-26 00:00:00
Dates are sorted: True


### Chronological Sorting Interpretation

The dataset has been successfully sorted by date from the earliest observation to the latest observation.

- The earliest date is **5 February 2010**.
- The latest date is **26 October 2012**.
- The result `Dates are sorted: True` confirms that the `date` column is arranged in chronological order.

This is important because the dataset contains time-based sales records. During model development, earlier observations should be used for training, while later observations should be reserved for testing.

Using a chronological split helps prevent data leakage and produces a more realistic evaluation of how the model may perform when predicting future weekly sales.

## Separating Features and Target

The target variable is `weekly_sales`.

All other suitable columns will be used as predictor features.

The original `date` column is retained temporarily for chronological splitting but will not be passed directly into the preprocessing transformer.

In [ ]:
target_column = "weekly_sales"

y = df[target_column].copy()

X = df.drop(
    columns=[target_column]
).copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (6435, 12)
Target shape: (6435,)


### Separating Features and Target Interpretation

The dataset was successfully divided into the predictor features (`X`) and the target variable (`y`).

The target variable is **`weekly_sales`**, which is the value the machine learning model will learn to predict.

All remaining columns were retained as predictor features to explain variations in weekly sales.

The output shows:

- **Features shape:** `(6435, 12)`
- **Target shape:** `(6435,)`

This means:

- The dataset contains **6,435 observations (rows)**.
- The feature matrix (`X`) contains **12 predictor variables**.
- The target vector (`y`) contains **6,435 weekly sales values**, with one target value corresponding to each observation.

The number of rows in both `X` and `y` is identical, confirming that every feature record has a matching target value. This ensures the dataset is correctly prepared for the chronological train-test splitting and model training stages.

## Why are there 12 features?

Originally, your dataset contained the target variable (weekly_sales) plus several predictor columns. After you removed only the target column using:

In [ ]:
target_column = "weekly_sales"

y = df[target_column].copy()

X = df.drop(
    columns=[target_column]
).copy()

In [ ]:
# Previewing the features and target

X.head()

,store,date,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,week_of_year,quarter,time_index
0,1,2010-02-05,0,42.31,2.572,211.096358,8.106,2010,2,5,1,0
1,2,2010-02-05,0,40.19,2.572,210.752605,8.324,2010,2,5,1,0
2,3,2010-02-05,0,45.71,2.572,214.424881,7.368,2010,2,5,1,0
3,4,2010-02-05,0,43.76,2.598,126.442065,8.623,2010,2,5,1,0
4,5,2010-02-05,0,39.70,2.572,211.653972,6.566,2010,2,5,1,0


In [ ]:
y.head()

0    1643690.90
1    2136989.46
2     461622.22
3    2135143.87
4     317173.10
Name: weekly_sales, dtype: float64

### Meaning
- X contains the information used to make predictions.
- y contains the actual weekly-sales values the model must learn to predict.

## Creating a chronological train-test split

### Choosing the split percentage

I will be using :
- 80% training data
- 20% testing data

In [ ]:
train_percentage = 0.80

split_index = int(
    len(df) * train_percentage
)

print("Split index:", split_index)

Split index: 5148


In [ ]:
test_percentage = 0.20

split_index = int(
    len(df) * test_percentage
)

print("Split index:", split_index)

Split index: 1287


With 6,435 records, the approximate split will be:

Training records: 5,148
Testing records: 1,287

In [ ]:
# Spliting the data chronologically

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1287, 12)
X_test shape: (5148, 12)
y_train shape: (1287,)
y_test shape: (5148,)


In [ ]:
# Confirming the train and test dates

print(
    "Training period:",
    X_train["date"].min(),
    "to",
    X_train["date"].max(),
)

print(
    "Testing period:",
    X_test["date"].min(),
    "to",
    X_test["date"].max(),
)


Training period: 2010-02-05 00:00:00 to 2010-08-20 00:00:00
Testing period: 2010-08-20 00:00:00 to 2012-10-26 00:00:00


### Chronological Train-Test Split Interpretation

The dataset was successfully divided into training and testing sets while preserving the chronological order of the observations.

**Training period**

- Start date: **2010-02-05**
- End date: **2010-08-20**

**Testing period**

- Start date: **2010-08-20**
- End date: **2012-10-26**

The training data contains the earlier observations, while the testing data contains the later observations. This approach simulates how the model would be used in a real-world forecasting environment, where historical data is used to predict future sales.

The latest training date is not later than the earliest testing date, confirming that no future information has been included in the training data.

Maintaining chronological order helps prevent **data leakage**, ensuring that the model is evaluated on genuinely unseen future observations and producing a more realistic estimate of its performance.

In [ ]:
# Checking for chronological leakage

latest_training_date = X_train["date"].max()
earliest_testing_date = X_test["date"].min()

print(
    "Latest training date:",
    latest_training_date,
)

print(
    "Earliest testing date:",
    earliest_testing_date,
)

print(
    "Chronological split valid:",
    latest_training_date <= earliest_testing_date,
)

Latest training date: 2010-08-20 00:00:00
Earliest testing date: 2010-08-20 00:00:00
Chronological split valid: True


### Chronological Leakage Validation Interpretation

The chronological leakage validation indicates that the dataset has been split in the correct chronological order.

The latest date in the training dataset is **2010-08-20**, while the earliest date in the testing dataset is also **2010-08-20**. Because the latest training date is not later than the earliest testing date, the validation returned **True**, confirming that the split follows chronological order.

However, the same date appears in both the training and testing datasets. This occurs because the split was performed using row positions, and multiple stores have sales records on the same date. As a result, some records for **2010-08-20** were assigned to the training set, while others were assigned to the testing set.

Although the validation passes and no future dates are included in the training data, a date-based split would be a more robust approach. Keeping all records from the same date together in either the training or testing dataset further reduces the risk of date-level information leakage and provides a more realistic evaluation of the model's performance on future sales data.

### Important observation about stores sharing the same date

Because each date contains records for multiple stores, an 80% row-based split can divide one date between training and testing.

The best approach is to split using unique dates.

So, Using the following approach below;

In [ ]:
# Improve the split using unique dates
# Replace the row-based split with this date-based version

unique_dates = (
    df["date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

date_split_index = int(
    len(unique_dates) * 0.80
)

training_dates = unique_dates.iloc[
    :date_split_index
]

testing_dates = unique_dates.iloc[
    date_split_index:
]

training_mask = df["date"].isin(
    training_dates
)

testing_mask = df["date"].isin(
    testing_dates
)

X_train = (
    df.loc[training_mask]
    .drop(columns=[target_column])
    .copy()
)

X_test = (
    df.loc[testing_mask]
    .drop(columns=[target_column])
    .copy()
)

y_train = (
    df.loc[training_mask, target_column]
    .copy()
)

y_test = (
    df.loc[testing_mask, target_column]
    .copy()
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5130, 12)
X_test shape: (1305, 12)
y_train shape: (5130,)
y_test shape: (1305,)


### Improved Chronological Train-Test Split Interpretation

The dataset was divided using its **unique dates** rather than individual row positions.

The resulting training dataset contains **5,130 records and 12 feature columns**, while the testing dataset contains **1,305 records and 12 feature columns**.

The target variables have matching numbers of observations:

- `y_train` contains **5,130 weekly-sales values**.
- `y_test` contains **1,305 weekly-sales values**.

This confirms that every feature row has a corresponding target value.

Using unique dates ensures that all Walmart stores recorded on the same weekly date remain together in one dataset. A particular week is therefore assigned entirely to either the training set or the testing set, rather than being divided between both.

This approach improves the earlier row-based split because it prevents the same date from appearing in both datasets. The model will be trained using earlier weeks and evaluated using later, unseen weeks, producing a more realistic estimate of how well it may predict future weekly sales.

The split is approximately **80% for training and 20% for testing**, although the exact record percentages may differ slightly because the split is based on unique dates rather than the total number of rows.

In [ ]:
# Validating the improved split

print(
    "Training period:",
    X_train["date"].min(),
    "to",
    X_train["date"].max(),
)

print(
    "Testing period:",
    X_test["date"].min(),
    "to",
    X_test["date"].max(),
)

shared_dates = set(
    X_train["date"]
).intersection(
    set(X_test["date"])
)

print(
    "Shared dates between training and testing:",
    len(shared_dates),
)

Training period: 2010-02-05 00:00:00 to 2012-04-06 00:00:00
Testing period: 2012-04-13 00:00:00 to 2012-10-26 00:00:00
Shared dates between training and testing: 0


### Chronological Split Interpretation

The dataset was split using unique dates rather than randomly selected rows.

All observations from earlier dates were placed in the training set, while observations from later dates were placed in the testing set.

No dates are shared between the two datasets. This prevents future information from leaking into the training data and provides a more realistic evaluation of the model's ability to predict future weekly sales.

### Removing the raw date from model inputs

The date is needed for checking chronology, but scikit-learn models cannot directly process raw datetime values.

In [ ]:
# Preserving dates separately

train_dates = X_train["date"].copy()
test_dates = X_test["date"].copy()

X_train = X_train.drop(
    columns=["date"]
)

X_test = X_test.drop(
    columns=["date"]
)

print("Date removed from model features.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Date removed from model features.
X_train shape: (5130, 11)
X_test shape: (1305, 11)


### Removing the Raw Date from Model Inputs — Interpretation

The preprocessing step successfully removed the original **`date`** column from the training and testing feature datasets.

The `date` column was useful for creating a chronological train-test split and verifying that the data remained in the correct time order. However, after the split was completed, the raw date was no longer required as a model input.

Instead, the date information was preserved separately in `train_dates` and `test_dates` so it can still be used later for reporting, visualization, or prediction analysis.

After removing the `date` column:

- **Training features (`X_train`)** contain **5,130 observations** and **11 predictor variables**.
- **Testing features (`X_test`)** contain **1,305 observations** and **11 predictor variables**.

The reduction from **12 features to 11 features** confirms that only the raw `date` column was removed, while all engineered date features (such as **year**, **month**, **week of year**, **quarter**, and **time index**) remain available for model training.

This is the correct preprocessing approach because most scikit-learn regression models cannot directly learn from raw datetime values. By removing the original date while keeping meaningful engineered time features, the dataset becomes fully compatible with machine learning algorithms and avoids introducing unnecessary information into the model.

### # Identifying feature groups

In [ ]:
# Checking the remaining columns

print("Remaining model features:")

for column in X_train.columns:
    print("-", column)



Remaining model features:
- store
- holiday_flag
- temperature
- fuel_price
- cpi
- unemployment
- year
- month
- week_of_year
- quarter
- time_index


### Define the categorical features

`store` represents separate store identities, not a continuous measurement.

`month` and `quarter` represent calendar categories.

In [ ]:
categorical_features = [
    "store",
    "month",
    "quarter",
]

categorical_features

['store', 'month', 'quarter']

### Defining Categorical and Binary Features — Interpretation

The predictor variables have been separated according to how they should be processed before model training.

#### Categorical features

The following variables were identified as categorical:

- `store`
- `month`
- `quarter`

Although these columns contain numbers, the numbers represent categories rather than continuous measurements.

For example:

- `store` identifies a particular Walmart store.
- `month` represents one of the twelve calendar months.
- `quarter` represents one of the four calendar quarters.

These features should therefore be processed using **one-hot encoding** so that the machine-learning model does not incorrectly assume that one category is numerically greater than another.

#### Binary feature

The `holiday_flag` column was identified as a binary feature.

Its values already have a meaningful numerical representation:

- `0` means a non-holiday week.
- `1` means a holiday week.

Because the column already contains only two machine-readable values, it does not require one-hot encoding and can be passed through the preprocessing pipeline unchanged.

The displayed outputs confirm that the feature groups were defined correctly:

- Categorical features: `['store', 'month', 'quarter']`
- Binary feature: `['holiday_flag']`

This separation will allow the preprocessing pipeline to apply the appropriate transformation to each type of feature.

In [ ]:
# Defining the binary feature

binary_features = [
    "holiday_flag",
]

binary_features

['holiday_flag']

In [ ]:
# Defining numerical features

numerical_features = [
    "temperature",
    "fuel_price",
    "cpi",
    "unemployment",
    "year",
    "week_of_year",
    "time_index",
]

numerical_features


['temperature',
 'fuel_price',
 'cpi',
 'unemployment',
 'year',
 'week_of_year',
 'time_index']

#### Interpetations 

The numerical feature group has been successfully defined for the preprocessing stage.

The selected numerical features are:

- Temperature
- Fuel Price
- Consumer Price Index (CPI)
- Unemployment Rate
- Year
- Week of Year
- Time Index

These features contain continuous or ordered numerical values that can be processed directly by the machine learning pipeline.

The output confirms that the column names in the dataset match the expected feature names, including `week_of_year`. Therefore, no changes are required before proceeding.

These numerical features will later be scaled using a numerical transformer to ensure they are on a comparable scale, helping regression models learn more effectively.

In [ ]:
# Automatically checking that every feature is assigned

assigned_features = (
    categorical_features
    + binary_features
    + numerical_features
)

unassigned_features = [
    column
    for column in X_train.columns
    if column not in assigned_features
]

unexpected_features = [
    column
    for column in assigned_features
    if column not in X_train.columns
]

print(
    "Unassigned dataset columns:",
    unassigned_features,
)

print(
    "Defined but missing columns:",
    unexpected_features,
)

Unassigned dataset columns: []
Defined but missing columns: []


## Feature Groups

The model features have been divided into three groups:

### Categorical features

- `store`
- `month`
- `quarter`

These values represent categories rather than continuous measurements. They will be transformed using one-hot encoding.

### Binary feature

- `holiday_flag`

This feature already contains numerical values of `0` and `1`, so it can pass through the preprocessing transformer unchanged.

### Numerical features

- `temperature`
- `fuel_price`
- `cpi`
- `unemployment`
- `year`
- `week_of_year`
- `time_index`

These features will be standardised so that measurements with different numerical scales can be processed more consistently by the machine-learning models.

#### Build the preprocessing transformer

In [ ]:
# Creating the numerical transformer

numerical_transformer = StandardScaler()

`StandardScaler` changes each numerical column so it is approximately centred around zero with a standard deviation of one.

The original data is not deleted or permanently changed. Scaling is applied inside the model workflow.

In [ ]:
# Creating the categorical transformer

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

#### Why handle_unknown="ignore"?

A future test or prediction dataset may contain a category that was not present while fitting the transformer.

This option prevents the pipeline from crashing.

#### Why sparse_output=False?

It produces a normal dense array that is easier for a beginner to inspect.

The dataset is small enough for this option.

In [ ]:
# Building the complete preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_transformer,
            numerical_features,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
)

## Preprocessing Transformer

The `ColumnTransformer` allows each feature group to receive the correct preprocessing operation.

- numerical features are standardised;
- categorical features are one-hot encoded;
- the binary holiday feature passes through unchanged;
- columns not explicitly assigned are excluded.

This transformer can later be combined with each regression model inside a scikit-learn pipeline.

#### Fitting and transforming the data

This is extremely important

Use fit_transform() on training data because the transformer must learn:

- training-data means;
- training-data standard deviations;
- training-data categories.

In [ ]:
# Fitting only on the training set

X_train_prepared = preprocessor.fit_transform(
    X_train
)

In [ ]:
# Transforming the test set

X_test_prepared = preprocessor.transform(
    X_test
)

We can use 
`preprocessor.fit_transform(X_test)`

Because that would allow the test data to influence the preprocessing and create data leakage.

In [ ]:
# Inspecting the transformed shapes

print(
    "Original X_train shape:",
    X_train.shape,
)

print(
    "Prepared X_train shape:",
    X_train_prepared.shape,
)

print(
    "Original X_test shape:",
    X_test.shape,
)

print(
    "Prepared X_test shape:",
    X_test_prepared.shape,
)

Original X_train shape: (5130, 11)
Prepared X_train shape: (5130, 69)
Original X_test shape: (1305, 11)
Prepared X_test shape: (1305, 69)


The number of columns will increase because one-hot encoding creates separate columns for store, month, and quarter categories.

### Recover the transformed feature names

In [ ]:
# Obtaining feature names

prepared_feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Number of prepared features:",
    len(prepared_feature_names),
)

prepared_feature_names[:20]

Number of prepared features: 69


array(['numerical__temperature', 'numerical__fuel_price',
       'numerical__cpi', 'numerical__unemployment', 'numerical__year',
       'numerical__week_of_year', 'numerical__time_index',
       'categorical__store_1', 'categorical__store_2',
       'categorical__store_3', 'categorical__store_4',
       'categorical__store_5', 'categorical__store_6',
       'categorical__store_7', 'categorical__store_8',
       'categorical__store_9', 'categorical__store_10',
       'categorical__store_11', 'categorical__store_12',
       'categorical__store_13'], dtype=object)

In [ ]:
# Converting the transformed arrays into DataFrames

X_train_prepared_df = pd.DataFrame(
    X_train_prepared,
    columns=prepared_feature_names,
    index=X_train.index,
)

X_test_prepared_df = pd.DataFrame(
    X_test_prepared,
    columns=prepared_feature_names,
    index=X_test.index,
)

X_train_prepared_df.head()

,numerical__temperature,numerical__fuel_price,numerical__cpi,numerical__unemployment,numerical__year,numerical__week_of_year,numerical__time_index,categorical__store_1,categorical__store_2,categorical__store_3,...,categorical__month_8,categorical__month_9,categorical__month_10,categorical__month_11,categorical__month_12,categorical__quarter_1,categorical__quarter_2,categorical__quarter_3,categorical__quarter_4,binary__holiday_flag
0,-0.830098,-1.540474,1.043783,-0.041372,-1.040454,-1.321231,-1.716923,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,-0.942822,-1.540474,1.034947,0.074674,-1.040454,-1.321231,-1.716923,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-0.649313,-1.540474,1.129343,-0.434223,-1.040454,-1.321231,-1.716923,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,-0.752999,-1.482088,-1.132278,0.233837,-1.040454,-1.321231,-1.716923,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,-0.968877,-1.540474,1.058116,-0.861143,-1.040454,-1.321231,-1.716923,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


#### This makes the transformed values easier to inspect and save.

## Validate the transformed data

In [ ]:
# Checking the missing values

print(
    "Missing values in prepared training data:",
    X_train_prepared_df.isnull().sum().sum(),
)

print(
    "Missing values in prepared testing data:",
    X_test_prepared_df.isnull().sum().sum(),
)

Missing values in prepared training data: 0
Missing values in prepared testing data: 0


In [ ]:
# Checking infinite values

training_infinite_values = np.isinf(
    X_train_prepared_df.to_numpy()
).sum()

testing_infinite_values = np.isinf(
    X_test_prepared_df.to_numpy()
).sum()

print(
    "Infinite values in training data:",
    training_infinite_values,
)

print(
    "Infinite values in testing data:",
    testing_infinite_values,
)

Infinite values in training data: 0
Infinite values in testing data: 0


In [ ]:
# Checking the target alignment

print(
    "Training rows match:",
    len(X_train_prepared_df) == len(y_train),
)

print(
    "Testing rows match:",
    len(X_test_prepared_df) == len(y_test),
)

Training rows match: True
Testing rows match: True


In [ ]:
# Checking for empty datasets

if X_train_prepared_df.empty:
    raise ValueError(
        "The prepared training dataset is empty."
    )

if X_test_prepared_df.empty:
    raise ValueError(
        "The prepared testing dataset is empty."
    )

print(
    "Prepared datasets passed the validation checks."
)

Prepared datasets passed the validation checks.


### Saving the preprocessing outputs

In [ ]:
# Resetting indexes before saving

X_train_prepared_df = (
    X_train_prepared_df
    .reset_index(drop=True)
)

X_test_prepared_df = (
    X_test_prepared_df
    .reset_index(drop=True)
)

y_train_save = (
    y_train
    .reset_index(drop=True)
    .rename("weekly_sales")
)

y_test_save = (
    y_test
    .reset_index(drop=True)
    .rename("weekly_sales")
)

train_dates_save = (
    train_dates
    .reset_index(drop=True)
    .rename("date")
)

test_dates_save = (
    test_dates
    .reset_index(drop=True)
    .rename("date")
)

In [ ]:
# Saving the training and testing datasets

X_train_path = (
    prepared_data_path
    / "X_train_prepared.csv"
)

X_test_path = (
    prepared_data_path
    / "X_test_prepared.csv"
)

y_train_path = (
    prepared_data_path
    / "y_train.csv"
)

y_test_path = (
    prepared_data_path
    / "y_test.csv"
)

train_dates_path = (
    prepared_data_path
    / "train_dates.csv"
)

test_dates_path = (
    prepared_data_path
    / "test_dates.csv"
)

X_train_prepared_df.to_csv(
    X_train_path,
    index=False,
)

X_test_prepared_df.to_csv(
    X_test_path,
    index=False,
)

y_train_save.to_csv(
    y_train_path,
    index=False,
)

y_test_save.to_csv(
    y_test_path,
    index=False,
)

train_dates_save.to_csv(
    train_dates_path,
    index=False,
)

test_dates_save.to_csv(
    test_dates_path,
    index=False,
)

print("Prepared datasets saved successfully.")

Prepared datasets saved successfully.


In [ ]:
from pathlib import Path
import pickle

# Current notebook folder
notebooks_path = Path.cwd()

# Move up until the project root is found
project_root = notebooks_path

while not (project_root / "models").exists():
    project_root = project_root.parent

# Models folder
models_path = project_root / "models"

# Make sure the folder exists
models_path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Models path:", models_path)

Project root: c:\Users\Udoch\walmart-sales-regression
Models path: c:\Users\Udoch\walmart-sales-regression\models


In [ ]:
# Saving the fitted preprocessor

preprocessor_path = (
    models_path
    / "walmart_preprocessor.pkl"
)

with open(preprocessor_path, "wb") as file:
    pickle.dump(
        preprocessor,
        file,
    )

print(
    "Preprocessor saved to:",
    preprocessor_path,
)

Preprocessor saved to: c:\Users\Udoch\walmart-sales-regression\models\walmart_preprocessor.pkl


In [ ]:
# Confirming all saved files exist

saved_files = [
    X_train_path,
    X_test_path,
    y_train_path,
    y_test_path,
    train_dates_path,
    test_dates_path,
    preprocessor_path,
]

for file_path in saved_files:
    print(
        file_path.name,
        "exists:",
        file_path.exists(),
    )

X_train_prepared.csv exists: True
X_test_prepared.csv exists: True
y_train.csv exists: True
y_test.csv exists: True
train_dates.csv exists: True
test_dates.csv exists: True
walmart_preprocessor.pkl exists: True


## Phase 7 Conclusion

The Walmart feature-engineered dataset has been prepared successfully for machine-learning modelling.

The following tasks were completed:

- loaded and validated the feature-engineered dataset;
- converted the date column back to a datetime data type;
- sorted the records chronologically;
- separated the predictor features from the `weekly_sales` target;
- divided the data into training and testing sets using unique dates;
- confirmed that no dates were shared between training and testing;
- removed the raw date column from the model inputs while preserving the date-based features;
- identified categorical, binary, and numerical feature groups;
- standardised numerical features;
- one-hot encoded categorical features;
- passed the binary holiday feature through unchanged;
- fitted the preprocessing transformer only on the training data;
- transformed the training and testing datasets consistently;
- confirmed that the prepared data contains no missing or infinite values;
- saved the prepared datasets and preprocessing transformer.

The data is now ready for Phase 8: Model Training and Comparison.

## Phase 7: Data Preprocessing Summary

The preprocessing phase was completed successfully.

The feature-engineered dataset was divided chronologically into training and testing datasets using unique dates. This ensured that all store records belonging to the same week remained together and prevented time-based data leakage.

The raw `date` column was preserved separately for chronological validation and removed from the machine-learning input features.

Categorical features were encoded, numerical features were scaled, and the binary holiday indicator was retained in its existing numerical format.

The following prepared datasets were saved in `data/prepared/`:

- `X_train_prepared.csv`
- `X_test_prepared.csv`
- `y_train.csv`
- `y_test.csv`
- `train_dates.csv`
- `test_dates.csv`

The fitted preprocessing transformer was also saved as a `.pkl` file in the `models/` directory.

The data is now ready for the model-training and model-comparison phase.

In [4]:
from pathlib import Path

print(Path.cwd())

c:\Users\Udoch\walmart-sales-regression\notebooks
